In [2]:
import os

In [3]:
%pwd

'c:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer'

In [6]:
## this is entity class
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class  DataValidationConfig:
    root_dir :Path
    data_path : Path
    model_path: Path
    tokenizer_path :Path
    metric_file_name:Path

In [7]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

from textSummarizer.entity import ModelEvaluationConfig

In [8]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath= PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name

        )

        return model_evaluation_config

In [9]:
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer
from datasets import load_from_disk
import evaluate
import torch
import pandas as pd
from tqdm import tqdm

c:\Users\Riyaz\OneDrive\Desktop\text_summarizer\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
import torch
from tqdm import tqdm
from datasets import load_from_disk
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from textSummarizer.entity import ModelEvaluationConfig


class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    def generate_batch_sized_chunks(self, list_of_elements, batch_size):

        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]


    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=2,
        device="cpu",
        column_text="dialogue",
        column_summary="summary"
    ):

        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_text],
                batch_size
            )
        )

        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_summary],
                batch_size
            )
        )

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=512,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=4,
                max_length=64
            )

            decoded_summaries = [
                tokenizer.decode(
                    s,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )
                for s in summaries
            ]

            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

        score = metric.compute()

        return score


    def evaluate(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"Using device: {device}")

        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)

        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt["test"].select(range(20)),
            metric=rouge_metric,
            model=model,
            tokenizer=tokenizer,
            batch_size=1,
            device=device
        )

        print("\nROUGE SCORE:\n")
        print(score)

In [15]:
try:
    config = ConfigurationManager()

    model_evaluation_config = config.get_model_evaluation_config()

    model_evaluation = ModelEvaluation(
        config=model_evaluation_config
    )

    model_evaluation.evaluate()

except Exception as e:
    raise e

[2026-05-11 22:29:25,137: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 22:29:25,141: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 22:29:25,144: INFO: common: created directory at: artifacts]
[2026-05-11 22:29:25,146: INFO: common: created directory at: artifacts/model_evaluation]
Using device: cpu


100%|██████████| 20/20 [00:52<00:00,  2.60s/it]

[2026-05-11 22:30:22,267: INFO: rouge_scorer: Using default tokenizer.]



ROUGE SCORE:

{'rouge1': np.float64(0.2524434038889983), 'rouge2': np.float64(0.053245818636932535), 'rougeL': np.float64(0.19904893652409), 'rougeLsum': np.float64(0.19823901868126562)}
